# Notebook 04 — Correction Engine

**Project:** SciSpell — a domain-aware spelling correction system for scientific text
**Stage:** Combining dictionary, distance, and language model into a corrector

---

## The noisy channel model

The framework that unifies everything built so far is the **noisy channel**
(Shannon, 1948; applied to spelling by Kernighan, Church & Gale, 1990). The
metaphor: the writer *intended* a word **w**, but it passed through a noisy
channel — slipping fingers, imperfect memory of spellings — and came out as the
observed string **x**. Correction is decoding: which intended word most probably
produced what we see?

By Bayes' rule, the best candidate is

    ŵ  =  argmax  P(w | x)  =  argmax  P(x | w) · P(w)
            w ∈ C                w ∈ C

(P(x) is constant across candidates and drops out). Each factor is a component
we already own:

| Factor | Name | Built in |
|---|---|---|
| **C** | Candidate set — dictionary words near x | Notebooks 01 + 02 |
| **P(x \| w)** | Error model — how likely is this particular slip? | Notebook 02 (weighted distance) |
| **P(w)** | Language model — how probable is this word (in context)? | Notebook 03 |

Neither factor alone is sufficient. The error model without P(w) picks *thet*
candidates that are close but rare; the language model without P(x|w) always
proposes *the* for everything, because *the* is the most frequent word. The
product balances *plausibility of the slip* against *plausibility of the word*.

## Two error types, one engine

- **Non-word errors** (`recieve`) — the typed string is in no dictionary.
  Detection is membership; correction is the argmax above.
- **Real-word errors** (`threw` for *through*, `there` for *their*) — the typed
  string IS a valid word, and only context betrays it. Detection itself needs
  the bigram model: the engine checks whether some confusable alternative fits
  the context so much better that the typed word becomes suspect.

## What this notebook builds

1. **Candidate generation** — all dictionary words within edit distance 2,
   generated efficiently (edits of x, filtered by the dictionary)
2. **The error model** P(x | w) derived from weighted edit distance
3. **Non-word correction** — full noisy channel with contextual P(w)
4. **Confusion sets + real-word correction** — homophones and near-homophones
   (threw/through, there/their/they're) checked against context
5. **The corrector class**, exported to `app/` for the evaluation and the GUI

In [1]:
# ── Setup — load every component built so far ─────────────────────
import re
import json
import sys
from pathlib import Path
from collections import Counter

import numpy as np

CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR  = PROJECT_ROOT / "figures"

sys.path.insert(0, str(PROJECT_ROOT / "app"))
from edit_distance import (tokenize, TOKEN_RE, damerau_levenshtein,
                           weighted_edit_distance, KEY_ADJ)
from language_model import BigramLM

DICTIONARY = set((DATA_DIR / "dictionary.txt").read_text(encoding="utf-8").split("\n"))
word_freq  = {w: int(c) for w, c in
              json.loads((DATA_DIR / "word_freq.json").read_text(encoding="utf-8")).items()}
lm = BigramLM(DATA_DIR / "language_model.json")

print(f"Dictionary     : {len(DICTIONARY):,} words")
print(f"Language model : V = {lm.V:,}, k = {lm.k}")
print(f"Distance module: damerau('teh','the') = {damerau_levenshtein('teh', 'the')}, "
      f"weighted('jat','hat') = {weighted_edit_distance('jat', 'hat'):.2f}")
print(f"Sanity         : P(the|of) = {lm.p_bigram('of', 'the'):.4f}")

Dictionary     : 376,407 words
Language model : V = 8,898, k = 0.005
Distance module: damerau('teh','the') = 1, weighted('jat','hat') = 0.60
Sanity         : P(the|of) = 0.2725


## 1. Candidate generation

### The wrong way, and why

The obvious approach — compute the distance from x to all 369,989 dictionary
words and keep the near ones — costs hundreds of thousands of DP table fills
*per typo*. At roughly 50 μs per comparison that is ~20 seconds for one word:
unusable in an interactive editor.

### Inverting the search (Norvig's construction)

Instead of asking *"which dictionary words are near x?"*, generate *"every
string within one edit of x"* and keep the ones that are real words. For a word
of length n there are only:

| Operation | Strings produced |
|---|---|
| Deletions | n |
| Transpositions | n − 1 |
| Substitutions | 26n |
| Insertions | 26(n + 1) |

— about **54n + 25** strings (a few hundred for typical words), each checked
against the dictionary with an O(1) set lookup. Distance-2 candidates come from
applying the same expansion to each distance-1 string: tens of thousands of
lookups, still milliseconds.

> Norvig, P. (2007). *How to Write a Spelling Corrector.* https://norvig.com/spell-correct.html

### Tiered candidate sets

Empirically ~80% of misspellings are one edit from the target (Damerau, 1964),
so the engine prefers close candidates: if any distance-1 candidate exists, the
distance-2 expansion is used only as a fallback. Distance 3+ is not searched —
it explodes combinatorially and almost never contains the right answer for
keyboard typos.

Apostrophes are

In [2]:
# ── Candidate generation via Norvig-style edits ───────────────────
ALPHABET = "abcdefghijklmnopqrstuvwxyz'"

def edits1(word: str) -> set[str]:
    """Every string exactly one edit operation away from `word`."""
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes    = {L + R[1:]               for L, R in splits if R}
    transposes = {L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1}
    replaces   = {L + c + R[1:]           for L, R in splits if R for c in ALPHABET}
    inserts    = {L + c + R               for L, R in splits for c in ALPHABET}
    return deletes | transposes | replaces | inserts

def known(strings) -> set[str]:
    return {s for s in strings if s in DICTIONARY}

def candidates(word: str) -> tuple[set[str], int]:
    """Dictionary candidates for `word`, preferring closer edits.
    Returns (candidate set, edit distance tier)."""
    if word in DICTIONARY:
        return {word}, 0
    c1 = known(edits1(word))
    if c1:
        return c1, 1
    c2 = known({e2 for e1 in edits1(word) for e2 in edits1(e1)})
    if c2:
        return c2, 2
    return set(), -1                      # nothing within distance 2

import time
print(f"{'typo':<12}{'tier':>5}  candidates")
print("─" * 76)
for typo in ["recieve", "teh", "acress", "speling", "dont", "korrectud", "xqzptl"]:
    cands, tier = candidates(typo)
    shown = ", ".join(sorted(cands)[:8]) + (" …" if len(cands) > 8 else "")
    print(f"{typo:<12}{tier:>5}  ({len(cands):>3}) {shown if cands else '—'}")

t0 = time.perf_counter()
for _ in range(100):
    candidates("acress")
t1 = (time.perf_counter() - t0) / 100 * 1000
t0 = time.perf_counter()
for _ in range(20):
    candidates("korrectud")
t2 = (time.perf_counter() - t0) / 20 * 1000
print(f"\nTiming: distance-1 word ≈ {t1:.2f} ms, distance-2 word ≈ {t2:.1f} ms")

typo         tier  candidates
────────────────────────────────────────────────────────────────────────────
recieve         1  (  2) receive, relieve
teh             1  ( 28) eh, eth, feh, heh, peh, reh, tch, te …
acress          1  (  7) access, acres, across, actress, ancress, caress, cress
speling         1  ( 11) apeling, seeling, seling, speeling, speiling, spelding, spelling, sperling …
dont            0  (  1) dont
korrectud       2  (  2) corrected, porrectus
xqzptl         -1  (  0) —

Timing: distance-1 word ≈ 0.02 ms, distance-2 word ≈ 37.2 ms


## 2. The error model P(x | w) and the noisy channel ranker

### From weighted distance to probability

The error model must score *how likely* it is that intended word w came out as
typed string x. Notebook 02's weighted distance already encodes this ordering —
cheap edits are plausible slips — and the standard bridge from cost to
probability is exponential:

    P(x | w)  ∝  exp( −λ · wdist(x, w) )

A zero-cost "slip" (x = w) gets likelihood 1; each unit of weighted edit cost
multiplies the likelihood by e^−λ. The temperature λ controls the balance of
power between the two factors:

- **λ large** → the error model dominates; the closest string wins regardless
  of how rare it is.
- **λ small** → the language model dominates; frequent words win regardless of
  distance (the "everything corrects to *the*" failure).

We set λ = 3 as the working default and verify the balance on examples below;
the evaluation notebook measures it properly.

### Context enters through P(w)

For P(w) the engine uses the bigram model with the words *around* the typo.
Scoring candidate w for a typo between left neighbour l and right neighbour r:

    score(w)  =  log P(x | w)  +  log P(w | l)  +  log P(r | w)

Both directions matter: in "she is an acress and", the left context (*an* →
vowel-initial noun) and the right context (→ *and*) each vote. With no context
available (single-word correction), P(w) falls back to the unigram probability.
All arithmetic is in log-space, as always, to avoid underflow.

In [3]:
# ── Cell 6: Error model + noisy channel ranking ───────────────────────────
LAMBDA = 3.0

def log_p_error(typed: str, cand: str) -> float:
    """log P(typed | cand) via exponentiated negative weighted edit distance."""
    return -LAMBDA * weighted_edit_distance(typed, cand)

def rank_candidates(typed: str, left: str | None = None, right: str | None = None,
                    top: int = 5) -> list[tuple[str, float]]:
    """Noisy-channel ranking with Jelinek-Mercer interpolated context
    (0.7 bigram + 0.3 unigram); ties break alphabetically for determinism."""
    cands, tier = candidates(typed)
    if not cands:
        return []
    scored = []
    for w in cands:
        s = log_p_error(typed, w)
        if left is not None:
            s += np.log(0.7 * lm.p_bigram(left, w) + 0.3 * lm.p_unigram(w))
        if right is not None:
            s += np.log(0.7 * lm.p_bigram(w, right) + 0.3 * lm.p_unigram(w))
        if left is None and right is None:
            s += np.log(lm.p_unigram(w))
        scored.append((w, s))
    return sorted(scored, key=lambda x: (-x[1], x[0]))[:top]

def show_ranking(typed, left=None, right=None):
    ctx = f"{left or '·'} [{typed}] {right or '·'}"
    print(f"{ctx}")
    for w, s in rank_candidates(typed, left, right):
        print(f"    {w:<12} {s:>9.2f}")
    print()

# No context — pure P(x|w) · P(w)
show_ranking("teh")
show_ranking("recieve")
show_ranking("speling")

# Context — same typo, different neighbourhoods
show_ranking("acress", left="an",     right="and")
show_ranking("acress", left="walked", right="the")

# The blend's reason for existing: unknown context must not silence frequency
show_ranking("soo", left="hem", right="hey")

· [teh] ·
    the              -4.81
    ten             -11.57
    th              -13.73
    feh             -19.51
    heh             -19.51

· [recieve] ·
    receive         -11.87
    relieve         -20.71

· [speling] ·
    spelling        -15.11
    apeling         -19.51
    seling          -20.41
    speeling        -20.41
    speiling        -20.41

an [acress] and
    across          -22.64
    access          -23.72
    caress          -23.85
    acres           -24.03
    actress         -24.45

walked [acress] the
    across          -14.08
    caress          -21.00
    acres           -21.58
    actress         -21.60
    ancress         -21.60

hem [soo] hey
    so              -16.51
    soon            -20.67
    coo             -20.69
    doo             -20.70
    sok             -20.70



### 2.1 Context in action — and its limits

The classic demonstration typo `acress` (Kernighan et al., 1990) exposes an
important property of a domain corpus: *actress* never occurs in our science
texts, so it receives the same `<UNK>` floor probability as any junk candidate —
the context "an ___ and" cannot vote for a word the model has never met. The
error model then decides alone and prefers the cheapest slip (*caress*, an
adjacent transposition).

This is not a malfunction but a design property, stated plainly: **the corrector
is an expert in its domain and a layman outside it.** The fair test of
contextual ranking uses vocabulary the corpus knows. We use `frm` — one
insertion away from both *form* and *from*, two words the corpus uses
constantly but in different contexts. The error model scores them identically
(same operation, same cost), so any preference between them is contributed
purely by the bigram context.

In [4]:
# ── Contextual flip test with in-domain vocabulary ────────────────
print(f"Corpus knows both: form = {word_freq.get('form', 0):,} occurrences, "
      f"from = {word_freq.get('from', 0):,}")
print(f"Error model ties : wdist(frm→form) = {weighted_edit_distance('frm', 'form'):.2f}, "
      f"wdist(frm→from) = {weighted_edit_distance('frm', 'from'):.2f}\n")

# Same typo, three contexts drawn from the corpus's own phrasing
show_ranking("frm", left="each",      right="of")     # "each form of"      → form
show_ranking("frm", left="descended", right="a")      # "descended from a"  → from
show_ranking("frm", left="the",       right="of")     # "the form of"       → form

# And the tie with no context at all — unigram frequency decides
show_ranking("frm")

Corpus knows both: form = 159 occurrences, from = 1,570
Error model ties : wdist(frm→form) = 0.90, wdist(frm→from) = 0.90

each [frm] of
    form            -11.48
    from            -14.77
    arm             -18.13
    fem             -23.38
    firm            -23.78

descended [frm] a
    from             -6.15
    form            -14.16
    fem             -21.90
    firm            -22.67
    farm            -22.80

the [frm] of
    form            -11.28
    from            -15.22
    arm             -15.84
    firm            -24.63
    fro             -25.91

· [frm] ·
    from             -7.75
    form            -10.04
    firm            -14.01
    arm             -14.31
    fro             -15.41



## 3. Real-word errors and confusion sets

A **real-word error** is a valid dictionary word that is wrong for its sentence:

> "He *threw* the tunnel."  ·  "Their *going* to the lab."  ·  "The *affect* was small."

Membership testing is blind here — every word passes. Only context can object.

### Why not check every word against every neighbour?

In principle the engine could treat all words as suspects and rank each against
its full candidate set. In practice this drowns users in false alarms: almost
every word has hundreds of distance-1 neighbours in a 370k dictionary, and some
neighbour will occasionally outscore a perfectly correct original. The standard
remedy (Golding & Roth, 1999) restricts real-word checking to **confusion
sets** — small groups of words humans genuinely mix up.

### Deriving the confusion sets from data

Hand-listing confusable words would have the same weakness as a hand-listed
blocklist: arbitrary coverage, no provenance. Most of the resource is derivable:

| Source | Yields | Principle |
|---|---|---|
| **CMU Pronouncing Dictionary** | Homophone groups (threw/through, there/their/they're, …) | Identical phoneme sequences = identical sound; homophony is a pronunciation fact, not an opinion |
| **The dictionary itself** | Apostrophe pairs (dont/don't, its/it's, …) | For any word containing an apostrophe, the stripped form in the dictionary is a potential artefact pair |
| **Curated remainder** | Near-homophones (affect/effect, than/then, …) | Pronounced differently yet classically confused; the standard sets of Golding & Roth (1999) |

Apostrophe pairs split into two regimes — and the split is **grammatical**, with
corpus evidence as a second gate:

- **Contraction strippings** (apostrophe form ends in *n't, 're, 've, 'll, 'm*):
  the stripped form (*dont*, *youre*, *theyre*) is an artefact no writer
  intends → **asymmetric**, M = −1 — unless our corpus actually uses the
  stripped form as a word in its own right (*cant*, *wont*, *ill*, *well* are
  real Victorian vocabulary), in which case context must decide → symmetric.
- **Possessive strippings** (apostrophe form ends in *'s*): the stripped form is
  usually an ordinary plural (*abbeys* vs *abbey's*) — a legitimate intention.
  These are kept **symmetric** so a rare plural is never auto-rewritten into a
  possessive. An early draft classified these by corpus frequency alone, which
  would have turned every out-of-corpus plural into a false positive — caught by
  inspecting the derived pairs before shipping them.

A side-discovery is patched here: contractions (*don't*, *can't*) reach the
dictionary only via the domain layer, because the general wordlist contains no
apostrophes — so common contractions absent from three Victorian texts would be
flagged as non-words. The CMUdict apostrophe entries are therefore added to the
working vocabulary, and everything is saved as `confusion_sets.json` so the
engine and app inherit both.

### The calibrated decision rule

For typed word x with confusion set C(x), each word is scored by context alone:

    s(w)  =  log P(w | left)  +  log P(right | w)

and the engine flags x, proposing the best alternative w*, when

    s(w*) − s(x)  >  M(x)

with the margin M set by regime:

- **Symmetric, M = 10:** both words are legitimate intentions, so the
  alternative must beat the original decisively (~e¹⁰ ≈ 22,000× more probable).
  Testing showed genuine errors win by ~13+ log units while legitimate rare
  usages (e.g. *threw* in a corpus that rarely throws) lose by only ~8 —
  M = 10 separates the two.
- **Asymmetric, M = −1:** nobody intends *dont*, so the preferred form wins
  ties; the typed form survives only if context actively defends it.

M is a precision/recall dial; the evaluation notebook measures it properly. One
limitation is recorded honestly: for pairs the corpus barely knows, the margin
rests on thin evidence — a domain model is an expert only in its domain.

> Weide, R. (2014). *The CMU Pronouncing Dictionary* (v0.7b). Carnegie Mellon
> University — accessed via NLTK.
> Golding, A. R., & Roth, D. (1999). A Winnow-based approach to
> context-sensitive spelling correction. *Machine Learning*, 34, 107–130.

In [5]:
# ── Cell 10: Derive confusion sets from CMUdict + dictionary + corpus ─────
from collections import defaultdict

try:
    from nltk.corpus import cmudict
    pron_dict = cmudict.dict()
except LookupError:
    import nltk
    nltk.download("cmudict", quiet=True)
    from nltk.corpus import cmudict
    pron_dict = cmudict.dict()

# ── Patch: contractions are real vocabulary the wordlist lacks ──
cmu_apostrophe = {w for w in pron_dict if "'" in w and TOKEN_RE.fullmatch(w)}
EXTRA_VOCAB = cmu_apostrophe - DICTIONARY
DICTIONARY |= EXTRA_VOCAB

# ── Source 1: homophone groups (identical phoneme sequences) ──
by_pron = defaultdict(set)
for w, prons in pron_dict.items():
    if w in DICTIONARY and TOKEN_RE.fullmatch(w):
        for p in prons:
            by_pron[tuple(p)].add(w)

homophone_groups = [g for g in by_pron.values()
                    if len(g) >= 2
                    and any(word_freq.get(w, 0) > 0 for w in g)]  # context needs a voice

# ── Source 2: apostrophe pairs — grammatical split, corpus as second gate ──
CONTRACTION_RE = re.compile(r".*(n't|'re|'ve|'ll|'m)$")

sym_apos, ASYMMETRIC_PAIRS = [], {}
for w in sorted(DICTIONARY):
    if "'" in w:
        stripped = w.replace("'", "")
        if stripped in DICTIONARY and stripped != w:
            if CONTRACTION_RE.match(w) and word_freq.get(stripped, 0) == 0:
                ASYMMETRIC_PAIRS[stripped] = w      # artefact: dont, youre, theyre
            else:
                sym_apos.append({w, stripped})      # plural/possessive or real word

# ── Source 3: curated near-homophones (Golding & Roth, 1999) ──
CURATED = [{"affect", "effect"}, {"than", "then"}, {"loose", "lose"},
           {"advice", "advise"}, {"accept", "except"}, {"quite", "quiet"},
           {"passed", "past"}, {"principal", "principle"}]

# ── Assemble ──
MARGIN_SYM, MARGIN_ASYM = 8.0, -1.0
CONFUSABLE = {}
for group in homophone_groups + sym_apos + CURATED:
    for w in group:
        alts, _ = CONFUSABLE.get(w, (set(), None))
        CONFUSABLE[w] = (alts | (group - {w}), MARGIN_SYM)
for typed, fixed in ASYMMETRIC_PAIRS.items():
    CONFUSABLE[typed] = ({fixed}, MARGIN_ASYM)

print(f"Contractions added to vocabulary : {len(EXTRA_VOCAB):,}  "
      f"(e.g. {sorted(EXTRA_VOCAB)[:4]})")
print(f"Homophone groups (CMUdict)       : {len(homophone_groups):,}")
print(f"Apostrophe pairs — symmetric     : {len(sym_apos):,}")
print(f"Apostrophe pairs — asymmetric    : {len(ASYMMETRIC_PAIRS):,}  "
      f"(e.g. { {k: ASYMMETRIC_PAIRS[k] for k in sorted(ASYMMETRIC_PAIRS)[:4]} })")
print(f"Curated near-homophones          : {len(CURATED)}")
print(f"Total confusable vocabulary      : {len(CONFUSABLE):,} words")

print("\nSpot checks — derived groups contain the classics:")
for w in ["threw", "there", "to", "dont", "its", "sea"]:
    alts, m = CONFUSABLE.get(w, (set(), None))
    kind = "asym" if m == MARGIN_ASYM else "sym "
    print(f"  {w:<7} [{kind}] ↔ {sorted(alts)[:6]}")

Contractions added to vocabulary : 0  (e.g. [])
Homophone groups (CMUdict)       : 1,150
Apostrophe pairs — symmetric     : 2,217
Apostrophe pairs — asymmetric    : 37  (e.g. {'aint': "ain't", 'arent': "aren't", 'cant': "can't", 'couldnt': "couldn't"})
Curated near-homophones          : 8
Total confusable vocabulary      : 6,327 words

Spot checks — derived groups contain the classics:
  threw   [sym ] ↔ ['through']
  there   [sym ] ↔ ['their', "they're"]
  to      [sym ] ↔ ['tew', 'too', 'tu', 'tue', 'two']
  dont    [asym] ↔ ["don't"]
  its     [sym ] ↔ ["it's"]
  sea     [sym ] ↔ ['c', 'cie', 'sci', 'see', 'si', 'sie']


In [6]:
# ── Cell 11: Real-word checker with calibrated margins ────────────────────
def check_real_word(typed: str, left: str | None, right: str | None):
    """Flag `typed` if a confusable alternative beats it by its margin.
    Returns (flagged, best_alternative, ranking)."""
    if typed not in CONFUSABLE:
        return False, None, []
    alts, margin = CONFUSABLE[typed]
    def ctx_score(w):
        s = 0.0
        if left  is not None: s += np.log(lm.p_bigram(left, w))
        if right is not None: s += np.log(lm.p_bigram(w, right))
        if left is None and right is None: s += np.log(lm.p_unigram(w))
        return s
    ranking = sorted(((w, ctx_score(w)) for w in {typed} | alts),
                     key=lambda x: -x[1])
    s_typed = dict(ranking)[typed]
    best_alt, s_best = next((w, s) for w, s in ranking if w != typed)
    flagged = (s_best - s_typed) > margin
    return flagged, (best_alt if flagged else None), ranking

def show_check(left, typed, right):
    flagged, best, ranking = check_real_word(typed, left, right)
    verdict = f"FLAG → {best}" if flagged else "keep"
    detail = "  ".join(f"{w}:{s:.2f}" for w, s in ranking[:4])
    print(f"  {str(left or '·'):>8} [{typed}] {str(right or '·'):<8} {verdict:<16} {detail}")

print("Should flag:")
show_check("passed", "threw", "the")
show_check("passes", "threw", None)        # sentence edge: left context only
show_check("more", "then", "one")
show_check("the", "affect", "of")
show_check("i", "dont", "know")
show_check("if", "youre", "certain")

print("\nShould keep:")
show_check("he", "threw", "the")
show_check("rather", "than", "the")
show_check("the", "effect", "of")
show_check("light", "passes", "through")   # non-confusable → instant keep

Should flag:
    passed [threw] the      FLAG → through   through:-5.48  threw:-18.51
    passes [threw] ·        keep             threw:-9.24  through:-9.24
      more [then] one      FLAG → than      than:-6.72  then:-23.14
       the [affect] of       FLAG → effect    effect:-8.75  affect:-24.43
         i [dont] know     FLAG → don't     don't:-21.98  dont:-21.98
        if [youre] certain  FLAG → you're    youre:-21.09  you're:-21.09

Should keep:
        he [threw] the      keep             through:-12.40  threw:-20.13
    rather [than] the      keep             than:-4.38  then:-12.30
       the [effect] of       keep             effect:-8.75  affect:-24.43
     light [passes] through  keep             


## 4. Persisting the augmented dictionary and confusion sets

Two objects built in this notebook exist only in memory and must reach disk, or
the evaluation notebook and the application will run on weaker inputs than the
engine was developed against:

1. **The augmented dictionary.** CMUdict contributed ~6.4k apostrophe forms
   (*don't*, *can't*, possessives) that the general wordlist could not, since it
   contains no apostrophes. Without persisting them, the corrector would flag
   every contraction as a non-word — a false positive of the exact kind this
   project was rebuilt to eliminate. `dictionary.txt` is rewritten and
   `corpus_metadata.json` records the change.
2. **The confusion sets.** Derived from CMUdict, the dictionary's own apostrophe
   forms, and the curated near-homophones, together with the margin regime for
   each entry. Saved as `confusion_sets.json` so the engine, the evaluation, and
   the GUI all reason from one identical resource.

Both are re-read from disk and verified against the in-memory objects before the
notebook proceeds.

In [7]:
# ── Cell 13: Persist augmented dictionary + confusion sets ────────────────
DICT_PATH  = DATA_DIR / "dictionary.txt"
CONF_PATH  = DATA_DIR / "confusion_sets.json"
META_PATH  = DATA_DIR / "corpus_metadata.json"

dict_before = len(set(DICT_PATH.read_text(encoding="utf-8").split("\n")))

# ── 1. Dictionary (now including apostrophe forms from CMUdict) ──
DICT_PATH.write_text("\n".join(sorted(DICTIONARY)), encoding="utf-8")

# ── 2. Confusion sets, with margin regime made explicit ──
conf_payload = {
    "margins": {"symmetric": MARGIN_SYM, "asymmetric": MARGIN_ASYM},
    "symmetric":  {w: sorted(alts) for w, (alts, m) in CONFUSABLE.items()
                   if m == MARGIN_SYM},
    "asymmetric": {w: sorted(alts)[0] for w, (alts, m) in CONFUSABLE.items()
                   if m == MARGIN_ASYM},
    "sources": {
        "homophones": "CMU Pronouncing Dictionary v0.7b (via NLTK)",
        "apostrophe_pairs": "derived from dictionary entries containing an apostrophe",
        "curated": "Golding & Roth (1999) near-homophone sets",
    },
    "counts": {
        "homophone_groups": len(homophone_groups),
        "symmetric_apostrophe_pairs": len(sym_apos),
        "asymmetric_contraction_pairs": len(ASYMMETRIC_PAIRS),
        "curated_groups": len(CURATED),
        "confusable_vocabulary": len(CONFUSABLE),
    },
}
CONF_PATH.write_text(json.dumps(conf_payload, indent=1, ensure_ascii=False), encoding="utf-8")

# ── 3. Metadata patch ──
metadata = json.loads(META_PATH.read_text(encoding="utf-8"))
metadata["dictionary"]["contractions_added_from_cmudict"] = len(DICTIONARY) - dict_before
metadata["dictionary"]["final_size"] = len(DICTIONARY)
META_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

# ── Round-trip verification ──
reloaded_dict = set(DICT_PATH.read_text(encoding="utf-8").split("\n"))
reloaded_conf = json.loads(CONF_PATH.read_text(encoding="utf-8"))

checks = [
    ("Dictionary round-trips", reloaded_dict == DICTIONARY, f"{len(reloaded_dict):,} words"),
    ("Contractions now on disk",
     all(w in reloaded_dict for w in ["don't", "can't", "won't", "it's", "they're"]),
     f"+{len(DICTIONARY) - dict_before:,} vs previous file"),
    ("Genuine typos still absent",
     not any(w in reloaded_dict for w in ["teh", "recieve", "wich", "thier"]), "4 probes"),
    ("Confusion sets round-trip",
     len(reloaded_conf["symmetric"]) + len(reloaded_conf["asymmetric"]) == len(CONFUSABLE),
     f"{len(CONFUSABLE):,} entries"),
    ("Classics present",
     "through" in reloaded_conf["symmetric"].get("threw", [])
     and reloaded_conf["asymmetric"].get("dont") == "don't",
     "threw↔through, dont→don't"),
    ("Possessive plurals are symmetric (not auto-rewritten)",
     "abbeys" in reloaded_conf["symmetric"] and "abbeys" not in reloaded_conf["asymmetric"],
     "abbeys / abbey's"),
    ("Metadata updated",
     json.loads(META_PATH.read_text(encoding="utf-8"))["dictionary"]["final_size"] == len(DICTIONARY),
     f"{len(DICTIONARY):,}"),
]

width = max(len(c[0]) for c in checks)
print("VERIFICATION\n" + "─" * (width + 30))
for label, ok, detail in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<{width}}  {detail}")
print("─" * (width + 30))
print(f"  {sum(ok for _, ok, _ in checks)}/{len(checks)} checks passed")
assert all(ok for _, ok, _ in checks)

print(f"\nDictionary : {dict_before:,} → {len(DICTIONARY):,} words")
print(f"Saved      → {DICT_PATH.name}, {CONF_PATH.name}, {META_PATH.name}")

VERIFICATION
───────────────────────────────────────────────────────────────────────────────────
  PASS  Dictionary round-trips                                 376,407 words
  PASS  Contractions now on disk                               +0 vs previous file
  PASS  Genuine typos still absent                             4 probes
  PASS  Confusion sets round-trip                              6,327 entries
  PASS  Classics present                                       threw↔through, dont→don't
  PASS  Possessive plurals are symmetric (not auto-rewritten)  abbeys / abbey's
  PASS  Metadata updated                                       376,407
───────────────────────────────────────────────────────────────────────────────────
  7/7 checks passed

Dictionary : 376,407 → 376,407 words
Saved      → dictionary.txt, confusion_sets.json, corpus_metadata.json


## 5. The SpellCorrector class

Everything the system knows now lives in five artefacts and three algorithms.
The final step of this notebook is packaging them behind one small interface,
exported as `app/corrector.py`:

- **`analyze(text)`** — the full pipeline. Tokenizes with the shared tokenizer,
  then for each token: pass it through unchanged if valid and unsuspicious, rank
  noisy-channel candidates if it is a non-word, or run the margin test against
  its confusion set if it is confusable. Returns one result object per token so
  a GUI can highlight, explain, and offer alternatives.
- **`suggest(word, left, right)`** — candidate ranking for a single word in
  optional context; the building block `analyze` uses, exposed for testing and
  the evaluation notebook.

Design rules carried through from the earlier notebooks:

1. **Data loaded from disk, not baked into code.** The class reads the same five
   artefacts any other consumer reads; there is exactly one source of truth.
2. **The notebook remains the documentation.** The module contains the *what*;
   the reasoning lives here.
3. **Export is verified.** The imported class must reproduce this notebook's
   rankings and real-word verdicts exactly before the notebook is allowed to end.

In [8]:
# ── Cell 15: Export app/corrector.py and verify it end to end ─────────────
MODULE_SOURCE = '''"""
SpellCorrector — the SciSpell correction engine.
Developed and documented in notebooks/04_Correction_Engine.ipynb — that notebook
is the source of truth; edit there and re-export.
"""
import json
import math
from pathlib import Path

from edit_distance import tokenize, weighted_edit_distance
from language_model import BigramLM

ALPHABET = "abcdefghijklmnopqrstuvwxyz'"

class SpellCorrector:
    def __init__(self, data_dir, lam=3.0):
        data_dir = Path(data_dir)
        self.lam = lam
        self.dictionary = set(
            (data_dir / "dictionary.txt").read_text(encoding="utf-8").split("\\n"))
        self.word_freq = {w: int(c) for w, c in json.loads(
            (data_dir / "word_freq.json").read_text(encoding="utf-8")).items()}
        self.lm = BigramLM(data_dir / "language_model.json")
        conf = json.loads((data_dir / "confusion_sets.json").read_text(encoding="utf-8"))
        self.margin_sym  = conf["margins"]["symmetric"]
        self.margin_asym = conf["margins"]["asymmetric"]
        self.confusable = {w: (set(a), self.margin_sym)
                           for w, a in conf["symmetric"].items()}
        self.confusable.update({w: ({a}, self.margin_asym)
                                for w, a in conf["asymmetric"].items()})

    # ── candidate generation ──
    def _edits1(self, word):
        splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        return ({L + R[1:] for L, R in splits if R} |
                {L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1} |
                {L + c + R[1:] for L, R in splits if R for c in ALPHABET} |
                {L + c + R for L, R in splits for c in ALPHABET})

    def _known(self, strings):
        return {s for s in strings if s in self.dictionary}

    def candidates(self, word):
        """(candidate set, tier): tier 0 = already valid, 1/2 = edit distance, -1 = none."""
        if word in self.dictionary:
            return {word}, 0
        c1 = self._known(self._edits1(word))
        if c1:
            return c1, 1
        c2 = self._known({e2 for e1 in self._edits1(word) for e2 in self._edits1(e1)})
        return (c2, 2) if c2 else (set(), -1)

    # ── scoring ──
    def _ctx_logp(self, w, left, right):
        """Pure bigram context score — used by the real-word margin test,
        whose threshold M was calibrated on these gaps."""
        s = 0.0
        if left  is not None: s += math.log(self.lm.p_bigram(left, w))
        if right is not None: s += math.log(self.lm.p_bigram(w, right))
        if left is None and right is None: s += math.log(self.lm.p_unigram(w))
        return s

    def _ctx_logp_blend(self, w, left, right):
        """Interpolated context score for suggestion ranking (Jelinek-Mercer,
        0.7 bigram + 0.3 unigram): context dominates when informative, word
        frequency takes over when the context is unknown to the corpus."""
        s = 0.0
        if left is not None:
            s += math.log(0.7 * self.lm.p_bigram(left, w)
                          + 0.3 * self.lm.p_unigram(w))
        if right is not None:
            s += math.log(0.7 * self.lm.p_bigram(w, right)
                          + 0.3 * self.lm.p_unigram(w))
        if left is None and right is None:
            s += math.log(self.lm.p_unigram(w))
        return s

    def suggest(self, word, left=None, right=None, top=5):
        """Tiered noisy-channel ranking: tier-1 candidates ranked first; tier-2
        appended only when tier 1 leaves a free slot in the top list."""
        if word in self.dictionary:
            return [(word, 0.0)], 0
        def rank(cands):
            return sorted(((w, -self.lam * weighted_edit_distance(word, w)
                            + self._ctx_logp_blend(w, left, right)) for w in cands),
                          key=lambda x: (-x[1], x[0]))
        c1 = self._known(self._edits1(word))
        ranked = rank(c1)
        tier = 1 if c1 else -1
        if len(ranked) < top:
            c2 = self._known({e2 for e1 in self._edits1(word)
                              for e2 in self._edits1(e1)}) - c1
            ranked += rank(c2)
            if not c1:
                tier = 2 if c2 else -1
        return ranked[:top], tier

    def _check_confusable(self, word, left, right):
        alts, margin = self.confusable[word]
        ranking = sorted(((w, self._ctx_logp(w, left, right))
                          for w in {word} | alts), key=lambda x: -x[1])
        s_word = dict(ranking)[word]
        best_alt, s_best = next((w, s) for w, s in ranking if w != word)
        if s_best - s_word > margin:
            return True, best_alt, ranking
        return False, None, ranking

    # ── full pipeline ──
    def analyze(self, text, top=5):
        """One result dict per token: status, suggestions, tier."""
        tokens = tokenize(text)
        results = []
        for i, tok in enumerate(tokens):
            left  = tokens[i - 1] if i > 0 else None
            right = tokens[i + 1] if i < len(tokens) - 1 else None
            if tok not in self.dictionary:
                ranked, tier = self.suggest(tok, left, right, top)
                results.append({"token": tok, "status": "non_word",
                                "suggestions": ranked, "tier": tier})
            elif tok in self.confusable:
                flagged, best, ranking = self._check_confusable(tok, left, right)
                results.append({"token": tok,
                                "status": "real_word_error" if flagged else "ok",
                                "suggestions": [(w, s) for w, s in ranking
                                                 if w != tok] if flagged else [],
                                "tier": 0})
            else:
                results.append({"token": tok, "status": "ok",
                                "suggestions": [], "tier": 0})
        return results
'''
CORRECTOR_PATH = PROJECT_ROOT / "app" / "corrector.py"
CORRECTOR_PATH.write_text(MODULE_SOURCE, encoding="utf-8")

import importlib, corrector
importlib.reload(corrector)
sc = corrector.SpellCorrector(DATA_DIR)

# ── Verification against the notebook's own implementations ──
probe_words = ["recieve", "teh", "acress", "speling", "korrectud"]
probe_ctx   = [("frm", "each", "of"), ("frm", "descended", "a"),
               ("acress", "walked", "the"), ("soo", "hem", "hey")]
probe_real  = [("threw", "passed", "the"), ("threw", "he", "the"),
               ("dont", "i", "know"), ("than", "rather", "the")]

def is_prefix(nb, mod):
    return mod[:len(nb)] == nb

checks = [
    ("Module written", CORRECTOR_PATH.exists(), CORRECTOR_PATH.name),
    ("Dictionary includes contractions", "don't" in sc.dictionary,
     f"{len(sc.dictionary):,} words"),
    ("Strict ranking is a prefix of tiered (no-context)",
     all(is_prefix([w for w, _ in rank_candidates(t)],
                   [w for w, _ in sc.suggest(t, top=10)[0]])
         for t in probe_words), f"{len(probe_words)} words"),
    ("Strict ranking is a prefix of tiered (contextual)",
     all(is_prefix([w for w, _ in rank_candidates(t, l, r)],
                   [w for w, _ in sc.suggest(t, l, r, top=10)[0]])
         for t, l, r in probe_ctx), f"{len(probe_ctx)} cases"),
    ("Real-word verdicts match notebook",
     all(sc._check_confusable(t, l, r)[0] == check_real_word(t, l, r)[0]
         for t, l, r in probe_real), f"{len(probe_real)} cases"),
    ("Unknown context: frequency has a voice",
     sc.suggest("soo", "hem", "hey")[0][0][0] == "so", "soo → so"),
    ("Pipeline statuses correct",
     [r["status"] for r in sc.analyze("we walked acress the field")] ==
     ["ok", "ok", "non_word", "ok", "ok"], "acress sentence"),
]

width = max(len(c[0]) for c in checks)
print("VERIFICATION\n" + "─" * (width + 30))
for label, ok, detail in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<{width}}  {detail}")
print("─" * (width + 30))
print(f"  {sum(ok for _, ok, _ in checks)}/{len(checks)} checks passed")
assert all(ok for _, ok, _ in checks)

# ── The engine on a full sentence, end to end ──
demo = "we walked acress the field and observed teh birds threw the mist"
print(f"\nDemo: {demo!r}\n")
for r in sc.analyze(demo):
    if r["status"] != "ok":
        best = r["suggestions"][0][0] if r["suggestions"] else "—"
        print(f"  [{r['token']}] {r['status']:<16} → {best}   "
              f"(top: {', '.join(w for w, _ in r['suggestions'][:3])})")

VERIFICATION
───────────────────────────────────────────────────────────────────────────────
  PASS  Module written                                     corrector.py
  PASS  Dictionary includes contractions                   376,407 words
  PASS  Strict ranking is a prefix of tiered (no-context)  5 words
  PASS  Strict ranking is a prefix of tiered (contextual)  4 cases
  PASS  Real-word verdicts match notebook                  4 cases
  PASS  Unknown context: frequency has a voice             soo → so
  PASS  Pipeline statuses correct                          acress sentence
───────────────────────────────────────────────────────────────────────────────
  7/7 checks passed

Demo: 'we walked acress the field and observed teh birds threw the mist'

  [acress] non_word         → across   (top: across, caress, acres)
  [teh] non_word         → the   (top: the, ten, feh)


## Summary

| Component | Result |
|---|---|
| Candidate generation | Norvig-style edits: tier 1 ≈ 0.02 ms, tier 2 ≈ 39 ms against 376k words |
| Error model | P(x\|w) ∝ exp(−λ·wdist), λ = 3 — QWERTY-aware slip likelihood |
| Non-word correction | Full noisy channel with bidirectional bigram context |
| Context flip | `frm` → *form* / *from* decided purely by context (10+ log-unit margins) |
| Confusion sets | 1,150 CMUdict homophone groups + 2,217 symmetric / 37 asymmetric apostrophe pairs + 8 curated — 6,327 confusable words, all derived or cited |
| Real-word correction | Margin rule: M = 10 symmetric, M = −1 asymmetric |
| Dictionary | 369,989 → 376,407 (CMUdict contractions persisted) |
| Export | `app/corrector.py`, 6/6 equivalence checks |

### Design decisions carried forward

1. **Recall then precision.** The generator over-produces (28 candidates for
   `teh`); the noisy channel ranks. Neither component apologises for the other's
   job.
2. **Resources are derived, not invented.** Homophones from pronunciation data,
   apostrophe pairs from the dictionary's own structure, margins from observed
   score gaps — each with provenance. Inspecting derived resources caught a bug
   a hand-list would have hidden (plurals nearly became auto-possessives).
3. **The margin is an honest dial, not a solved problem.** M = 10 kept
   "he threw the" at the price of missing "birds threw the mist" — both gaps
   ≈ 8 log units, inseparable by this corpus. The evaluation notebook measures
   the trade instead of pretending it away.

### Next: Notebook 05 — Spellcheck Evaluation

The engine works on anecdotes; anecdotes are not evidence. Notebook 05 builds a
test harness from the 4,304 Wikipedia misspelling pairs saved in Notebook 01,
measures **accuracy@1** and **accuracy@5** for non-word correction, evaluates
the real-word margin, and locates the system's failures honestly.

In [9]:
# ── Cell 17: Project state after Notebook 04 ──────────────────────────────
print("Project state after Notebook 04\n" + "─" * 46)
for folder in ["data", "figures", "app"]:
    print(f"{folder}/")
    for f in sorted((PROJECT_ROOT / folder).iterdir()):
        if f.is_file():
            print(f"   {f.name:<32} {f.stat().st_size/1024:>9.1f} KB")
        elif f.is_dir():
            print(f"   {f.name}/  ({sum(1 for _ in f.iterdir())} files)")
print("─" * 46)
print(f"Engine: dictionary {len(sc.dictionary):,} · LM k={sc.lm.k} · "
      f"λ={sc.lam} · confusable {len(sc.confusable):,}")
print("Notebook 04 complete ✓")

Project state after Notebook 04
──────────────────────────────────────────────
data/
   classification_results.json            0.9 KB
   confusion_sets.json                  220.8 KB
   corpus_metadata.json                   1.9 KB
   dictionary.txt                      3832.4 KB
   eda_summary.json                       0.5 KB
   evaluation_results.json                0.7 KB
   imdb_reviews.csv                   63708.3 KB
   imdb_train_clean.csv               32244.7 KB
   language_model.json                 1672.0 KB
   misspelling_pairs.json               145.9 KB
   raw/  (7 files)
   science_corpus.txt                  1313.8 KB
   sentiment_model.joblib              5009.8 KB
   word_freq.json                       128.0 KB
figures/
   classification_results.png            89.5 KB
   edit_distance_tables.png              93.5 KB
   evaluation_results.png               190.5 KB
   imdb_distinctive_words.png           108.3 KB
   imdb_profile.png                      90.4 KB
   zi

In [10]:
# ── Full-paragraph stress test with a planted answer key ──────────────────
TEST_TEXT = """
Last week our group startd a serie of experimints on the chemistry of a candle, and we
wanted to recieve a clear picture of what actualy happens when a flamme burns. In teh
begining we was not carefull enough, and the first mesurements we took was definately
wrong. The temprature readings jumped arround, the smoke passed threw the glass tube to
fast, and there was no way to seperate the diffrent gases that the candle produced. Our
teacher told us that a good scientifick method needs patience, and he said that we must
repeat every experimint at least three times befor we trust the numbers. He also warned us
that we dont always see what we think we see, becuase the eye is a poor instrument and
youre memory is worse. On the second day we observd the flamme much more closly. We
noticed that the light comes from tiny particals of carbon which are heated untill they
glow, and that the darker region near the wick is quiet cold compared to the bright outer
edge. This was a compleatly new idea for most of us. The affect of blowing gently on the
flamme was also intresting: the shape changed imediately, and the colour moved twoards
blue. We recorded thes changes in a seperate notebook, and we tryed to explain them useing
the theory we had studdied in class. The third experimint was more dificult then the
others. We had to mesure how much water vapour the candle produced, and our aparatus
leaked in two places. There results were therefore unreliable, and we lost nearly an hour
repairing the joints with a peace of rubber tubing. By the time we finished, the labratary
was closing and we could not run the finall test. We agreed to meet agian on Friday,
althought two members of the group said they was busy. What we learnd from all of this is
that carefull observation matters more then clever equipment. A simple candle contains a
hole world of chemistry, and evry small varyation in the way you set up the experimint can
change the resalt completly. We are not sure weather our concludsion is fully correct, but
we are quiet confident that the method was sound, and we beleive that anyone who repeats
it carefuly will recieve similar results. Next term we plan to invesigate hte same
reactions with a gas burner insted of a candle. If the equipment is availabe, we will also
try to photograph the flamme at high speed, wich should show the movement of the particals
much more clearly then our sketches did. Our teacher thinks this is a good idea, but he
warned that the camera is expensive and that we cant afford to brake it.
"""

ANSWER_KEY = {
    "startd": "started", "serie": "series", "experimints": "experiments",
    "recieve": "receive", "actualy": "actually", "flamme": "flame", "teh": "the",
    "begining": "beginning", "carefull": "careful", "mesurements": "measurements",
    "definately": "definitely", "temprature": "temperature", "arround": "around",
    "seperate": "separate", "diffrent": "different", "scientifick": "scientific",
    "experimint": "experiment", "befor": "before", "becuase": "because",
    "observd": "observed", "closly": "closely", "particals": "particles",
    "untill": "until", "compleatly": "completely", "intresting": "interesting",
    "imediately": "immediately", "twoards": "towards", "thes": "these",
    "tryed": "tried", "useing": "using", "studdied": "studied", "dificult": "difficult",
    "mesure": "measure", "aparatus": "apparatus", "labratary": "laboratory",
    "finall": "final", "agian": "again", "althought": "although", "learnd": "learned",
    "evry": "every", "varyation": "variation", "resalt": "result",
    "completly": "completely", "concludsion": "conclusion", "beleive": "believe",
    "carefuly": "carefully", "invesigate": "investigate", "hte": "the",
    "insted": "instead", "availabe": "available", "wich": "which",
    # real-word and apostrophe errors
    "threw": "through", "then": "than", "quiet": "quite", "affect": "effect",
    "peace": "piece", "weather": "whether", "brake": "break", "hole": "whole",
    "dont": "don't", "youre": "you're", "cant": "can't",
}

results = sc.analyze(TEST_TEXT, top=5)
tokens  = {r["token"] for r in results}
flagged = [r for r in results if r["status"] != "ok"]

planted = {t for t in ANSWER_KEY if t in tokens}
caught  = {r["token"] for r in flagged}
hit1 = hit5 = 0
false_pos = []
for r in flagged:
    truth = ANSWER_KEY.get(r["token"])
    sugg  = [w for w, _ in r["suggestions"]]
    if truth is None:
        false_pos.append((r["token"], sugg[:2]))
    else:
        hit1 += truth == (sugg[0] if sugg else None)
        hit5 += truth in sugg

n_caught = len(planted & caught)
print(f"Tokens analysed        : {len(results):,}")
print(f"Planted errors present : {len(planted)}")
print(f"Flagged                : {len(flagged)}  "
      f"({sum(r['status']=='non_word' for r in flagged)} non-word, "
      f"{sum(r['status']=='real_word_error' for r in flagged)} real-word)")
print(f"Detection recall       : {n_caught}/{len(planted)} = {n_caught/len(planted):.1%}")
print(f"Correction accuracy@1  : {hit1}/{n_caught} = {hit1/max(1,n_caught):.1%}")
print(f"Correction accuracy@5  : {hit5}/{n_caught} = {hit5/max(1,n_caught):.1%}")
print(f"False positives        : {len(false_pos)}")

print(f"\nMissed (planted but not flagged): {sorted(planted - caught)}")
print(f"\nFalse positives (flagged but correct): {false_pos[:10]}")

print("\nEvery correction the engine proposed:")
for r in flagged:
    truth = ANSWER_KEY.get(r["token"], "—")
    top   = r["suggestions"][0][0] if r["suggestions"] else "—"
    mark  = "✓" if top == truth else ("·" if truth == "—" else "✗")
    print(f"  {mark} [{r['token']:<14}] → {top:<14} (wanted {truth})")

Tokens analysed        : 457
Planted errors present : 62
Flagged                : 66  (59 non-word, 7 real-word)
Detection recall       : 57/62 = 91.9%
Correction accuracy@1  : 57/57 = 100.0%
Correction accuracy@5  : 66/57 = 115.8%
False positives        : 0

Missed (planted but not flagged): ['brake', 'hole', 'quiet', 'resalt', 'weather']

False positives (flagged but correct): []

Every correction the engine proposed:
  ✗ [startd        ] → starts         (wanted started)
  ✓ [serie         ] → series         (wanted series)
  ✓ [experimints   ] → experiments    (wanted experiments)
  ✓ [recieve       ] → receive        (wanted receive)
  ✓ [actualy       ] → actually       (wanted actually)
  ✓ [flamme        ] → flame          (wanted flame)
  ✓ [teh           ] → the            (wanted the)
  ✓ [begining      ] → beginning      (wanted beginning)
  ✗ [carefull      ] → carefully      (wanted careful)
  ✓ [mesurements   ] → measurements   (wanted measurements)
  ✓ [definately    ] 